<a href="https://colab.research.google.com/github/Pri-codes-10/Leviathan_ml_practice/blob/main/leviathan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
train = pd.read_csv("train.csv")

print(train.shape)
train.head()

(300000, 28)


,sequence_id,timestep,sensor_0,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,...,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,level
0,0,1,0.797988,4.043916,0.855621,4.557594,8.341861,2.861494,0.153825,0.245543,...,0.021982,1.568427,0.211843,0.963500,3.049135,4.506402,8.061554,3.406531,6.500810,2.0
1,0,2,3.542572,1.454165,4.397192,3.334686,3.895221,0.946766,4.982987,4.270716,...,0.572948,2.600176,0.869560,0.020646,7.714765,0.136637,0.118311,2.517115,0.624052,2.0
2,0,3,1.825843,4.878263,4.508513,0.489621,2.400877,1.201121,4.050717,1.707504,...,1.174360,1.718682,1.042756,0.444506,6.758080,2.639249,2.850394,3.161774,7.647573,2.0
3,0,4,2.315402,2.839605,4.443792,1.602136,8.462071,2.723718,3.727529,2.289420,...,2.201044,2.512720,1.623894,1.856155,0.375670,4.648269,7.499120,3.298865,1.033686,2.0
4,0,5,6.792719,2.199233,3.239968,1.119951,8.425677,0.652766,3.595458,0.895690,...,0.256178,1.849288,1.598129,0.509279,7.616072,4.289352,8.252672,4.073928,3.592834,2.0


In [ ]:
sensor_cols = [col for col in train.columns if "sensor_" in col]

print(len(sensor_cols))

25


In [ ]:
X = []
y = []

for seq_id, group in train.groupby("sequence_id"):

    group = group.sort_values("timestep")

    X.append(group[sensor_cols].values)

    y.append(group["level"].iloc[0])

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(30000, 10, 25)
(30000,)


In [ ]:
scaler = StandardScaler()

X_flat = X.reshape(-1, X.shape[-1])

X_flat = scaler.fit_transform(X_flat)

X = X_flat.reshape(X.shape)

print(X.shape)

(30000, 10, 25)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
class LeviathanDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_dataset = LeviathanDataset(X_train, y_train)
val_dataset = LeviathanDataset(X_val, y_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

In [ ]:
import math

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=100):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]

In [ ]:
class CNNTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=25,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.bn1 = nn.BatchNorm1d(64)

        self.conv2 = nn.Conv1d(
            in_channels=64,
            out_channels=128,
            kernel_size=3,
            padding=1
        )

        self.bn2 = nn.BatchNorm1d(128)

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(0.3)

        self.pos_encoder = PositionalEncoding(128)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            dim_feedforward=512,
            dropout=0.3,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=4
        )

        self.attention = nn.Sequential(
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 4)
        )

    def forward(self, x):

        x = x.permute(0, 2, 1)

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)

        x = x.permute(0, 2, 1)

        x = self.pos_encoder(x)

        x = self.transformer(x)

        weights = torch.softmax(
            self.attention(x),
            dim=1
        )

        x = torch.sum(
            x * weights,
            dim=1
        )

        out = self.classifier(x)

        return out

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [ ]:
model = CNNTransformer().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    patience=3,
    factor=0.5
)

In [ ]:
def train_one_epoch(model, loader):

    model.train()

    total_loss = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(
            outputs,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def evaluate(model, loader):

    model.eval()

    preds = []
    labels = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)

            outputs = model(X_batch)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            preds.extend(
                predictions.cpu().numpy()
            )

            labels.extend(
                y_batch.numpy()
            )

    acc = accuracy_score(
        labels,
        preds
    )

    return acc

In [ ]:
best_acc = 0

for epoch in range(50):

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    val_acc = evaluate(
        model,
        val_loader
    )

    scheduler.step(val_acc)

    print(
        f"Epoch {epoch+1} | "
        f"Loss {train_loss:.4f} | "
        f"Val Acc {val_acc:.4f}"
    )

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        print("Model Saved")

Epoch 1 | Loss 1.3785 | Val Acc 0.3060
Model Saved
Epoch 2 | Loss 1.3760 | Val Acc 0.3060
Epoch 3 | Loss 1.3761 | Val Acc 0.3060
Epoch 4 | Loss 1.3745 | Val Acc 0.3060
Epoch 5 | Loss 1.3635 | Val Acc 0.3092
Model Saved
Epoch 6 | Loss 1.3183 | Val Acc 0.3555
Model Saved
Epoch 7 | Loss 1.2833 | Val Acc 0.3648
Model Saved
Epoch 8 | Loss 1.2607 | Val Acc 0.3727
Model Saved
Epoch 9 | Loss 1.2434 | Val Acc 0.3785
Model Saved
Epoch 10 | Loss 1.2205 | Val Acc 0.3827
Model Saved
Epoch 11 | Loss 1.2036 | Val Acc 0.3860
Model Saved
Epoch 12 | Loss 1.1840 | Val Acc 0.3872
Model Saved
Epoch 13 | Loss 1.1650 | Val Acc 0.3845
Epoch 14 | Loss 1.1442 | Val Acc 0.3895
Model Saved
Epoch 15 | Loss 1.1268 | Val Acc 0.3932
Model Saved
Epoch 16 | Loss 1.1016 | Val Acc 0.3905
Epoch 17 | Loss 1.0837 | Val Acc 0.3860
Epoch 18 | Loss 1.0610 | Val Acc 0.3828
Epoch 19 | Loss 1.0429 | Val Acc 0.3885
Epoch 20 | Loss 1.0101 | Val Acc 0.3793
Epoch 21 | Loss 0.9930 | Val Acc 0.3842
Epoch 22 | Loss 0.9823 | Val Acc 0.37

In [ ]:
model.load_state_dict(
    torch.load(
        "best_model.pth"
    )
)

<All keys matched successfully>

In [ ]:
final_acc = evaluate(
    model,
    val_loader
)

print(
    "Validation Accuracy:",
    final_acc
)

Validation Accuracy: 0.39316666666666666


In [ ]:
print(train.head())
print(train.columns)
print(train['level'].value_counts())

   sequence_id  timestep  sensor_0  sensor_1  sensor_2  sensor_3  sensor_4  \
0            0         1  0.797988  4.043916  0.855621  4.557594  8.341861   
1            0         2  3.542572  1.454165  4.397192  3.334686  3.895221   
2            0         3  1.825843  4.878263  4.508513  0.489621  2.400877   
3            0         4  2.315402  2.839605  4.443792  1.602136  8.462071   
4            0         5  6.792719  2.199233  3.239968  1.119951  8.425677   

   sensor_5  sensor_6  sensor_7  ...  sensor_16  sensor_17  sensor_18  \
0  2.861494  0.153825  0.245543  ...   0.021982   1.568427   0.211843   
1  0.946766  4.982987  4.270716  ...   0.572948   2.600176   0.869560   
2  1.201121  4.050717  1.707504  ...   1.174360   1.718682   1.042756   
3  2.723718  3.727529  2.289420  ...   2.201044   2.512720   1.623894   
4  0.652766  3.595458  0.895690  ...   0.256178   1.849288   1.598129   

   sensor_19  sensor_20  sensor_21  sensor_22  sensor_23  sensor_24  level  
0   0.963500   

In [ ]:
sequence_labels = train.groupby("sequence_id")["level"].first()

print(sequence_labels.value_counts())

level
3.0    9179
0.0    7946
2.0    6836
1.0    6039
Name: count, dtype: int64


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np

features = []
labels = []

for seq_id, group in train.groupby("sequence_id"):

    group = group.sort_values("timestep")

    seq = group[sensor_cols].values

    feat = []

    feat.extend(seq.mean(axis=0))
    feat.extend(seq.std(axis=0))
    feat.extend(seq.min(axis=0))
    feat.extend(seq.max(axis=0))
    feat.extend(seq[-1] - seq[0])

    features.append(feat)

    labels.append(group["level"].iloc[0])

X = np.array(features)
y = np.array(labels)

X_train,X_val,y_train,y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train,y_train)

pred = model.predict(X_val)

print("Accuracy:",accuracy_score(y_val,pred))

Accuracy: 0.30733333333333335


In [ ]:
print(train.groupby("sequence_id").size().unique())

[10]


In [ ]:
bad = 0

for seq_id, g in train.groupby("sequence_id"):
    if g["level"].nunique() != 1:
        bad += 1

print(bad)

0


In [ ]:
for c in range(4):
    subset = train[train["level"] == c]

    print(f"\nClass {c}")
    print(subset[sensor_cols].mean().head())


Class 0
sensor_0    3.646547
sensor_1    3.221240
sensor_2    3.237472
sensor_3    2.867485
sensor_4    5.533210
dtype: float64

Class 1
sensor_0    3.633958
sensor_1    3.230788
sensor_2    3.249941
sensor_3    2.870429
sensor_4    5.552591
dtype: float64

Class 2
sensor_0    3.624380
sensor_1    3.180438
sensor_2    3.219183
sensor_3    2.846870
sensor_4    5.513125
dtype: float64

Class 3
sensor_0    3.638277
sensor_1    3.199184
sensor_2    3.228336
sensor_3    2.851698
sensor_4    5.506874
dtype: float64


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = train[sensor_cols]
y = train["level"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    tree_method="hist",
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))

Accuracy: 0.29158333333333336


In [ ]:
import numpy as np

sensor_cols = [c for c in train.columns if c.startswith("sensor_")]

features = []
labels = []

for seq_id, g in train.groupby("sequence_id"):

    g = g.sort_values("timestep")

    seq = g[sensor_cols].values

    feat = seq.flatten()

    features.append(feat)

    labels.append(g["level"].iloc[0])

X = np.array(features)
y = np.array(labels)

print(X.shape)

(30000, 250)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

model = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    n_estimators=1000,
    max_depth=10,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))

Accuracy: 0.2985


In [ ]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.05
)

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW


In [ ]:
class MyModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc = nn.Linear(25, 4)

    def forward(self, x):
        return self.fc(x.mean(dim=1))

model = MyModel()

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

In [ ]:
print(type(model))

<class '__main__.MyModel'>


In [ ]:
print(X_train.shape)
print(y_train.shape)

(24000, 250)
(24000,)


In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score
import copy

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# ============================================================
# 3. LOAD DATA
# ============================================================

train = pd.read_csv("train.csv")

print(train.shape)
print(train.head())

# ============================================================
# 4. SENSOR COLUMNS
# ============================================================

sensor_cols = [
    c for c in train.columns
    if c.startswith("sensor_")
]

print("Sensors:", len(sensor_cols))

# ============================================================
# 5. BUILD SEQUENCES
# ============================================================

X = []
y = []

for seq_id, group in train.groupby("sequence_id"):

    group = group.sort_values("timestep")

    X.append(
        group[sensor_cols].values
    )

    y.append(
        int(group["level"].iloc[0])
    )

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

# EXPECTED
# (30000,10,25)
# (30000,)

# ============================================================
# 6. SCALE FEATURES
# ============================================================

scaler = StandardScaler()

X_flat = X.reshape(-1, 25)

X_flat = scaler.fit_transform(X_flat)

X = X_flat.reshape(X.shape)

# ============================================================
# 7. TRAIN VALIDATION SPLIT
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)

# ============================================================
# 8. DATASET
# ============================================================

class LeviathanDataset(Dataset):

    def __init__(self, X, y=None):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = y

        if y is not None:

            self.y = torch.tensor(
                y,
                dtype=torch.long
            )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        if self.y is None:
            return self.X[idx]

        return (
            self.X[idx],
            self.y[idx]
        )

# ============================================================
# 9. DATALOADERS
# ============================================================

train_dataset = LeviathanDataset(
    X_train,
    y_train
)

val_dataset = LeviathanDataset(
    X_val,
    y_val
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False
)

xb, yb = next(iter(train_loader))

print(xb.shape)
print(yb.shape)

# EXPECTED
# torch.Size([256,10,25])

# ============================================================
# 10. MODEL
# ============================================================

class BiLSTMAttention(nn.Module):

    def __init__(self):

        super().__init__()

        self.input_projection = nn.Linear(
            25,
            128
        )

        self.lstm1 = nn.LSTM(
            input_size=128,
            hidden_size=128,
            batch_first=True,
            bidirectional=True
        )

        self.lstm2 = nn.LSTM(
            input_size=256,
            hidden_size=128,
            batch_first=True,
            bidirectional=True
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=256,
            num_heads=8,
            batch_first=True
        )

        self.norm = nn.LayerNorm(256)

        # ATTENTION POOLING

        self.attn_pool = nn.Sequential(

            nn.Linear(
                256,
                128
            ),

            nn.Tanh(),

            nn.Linear(
                128,
                1
            )
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                256,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.4),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                4
            )
        )

    def forward(self, x):

        x = self.input_projection(x)

        x, _ = self.lstm1(x)

        x, _ = self.lstm2(x)

        attn_out, _ = self.attention(
            x,
            x,
            x
        )

        x = self.norm(
            x + attn_out
        )

        weights = torch.softmax(
            self.attn_pool(x),
            dim=1
        )

        x = torch.sum(
            x * weights,
            dim=1
        )

        return self.classifier(x)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)


# ============================================================
# CLASS WEIGHTS
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

weights = torch.tensor(
    weights,
    dtype=torch.float32
).to(device)


# ============================================================
# MODEL INIT
# ============================================================

model = BiLSTMAttention().to(device)

criterion = nn.CrossEntropyLoss(
    weight=weights,
    label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50
)


# ============================================================
# TRAIN FUNCTION
# ============================================================

def train_epoch():

    model.train()

    total_loss = 0

    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(
            outputs,
            y_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

        preds = outputs.argmax(1)

        correct += (
            preds == y_batch
        ).sum().item()

        total += y_batch.size(0)

    acc = correct / total

    return total_loss / len(train_loader), acc


# ============================================================
# VALIDATION FUNCTION
# ============================================================

def validate(model, loader):

    model.eval()

    preds = []
    labels = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)

            outputs = model(X_batch)

            pred = outputs.argmax(1)

            preds.extend(
                pred.cpu().numpy()
            )

            labels.extend(
                y_batch.numpy()
            )

    return accuracy_score(
        labels,
        preds
    )


# ============================================================
# TRAINING LOOP
# ============================================================

def train_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    total_loss = 0

    correct = 0
    total = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(
            outputs,
            y_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

        preds = outputs.argmax(1)

        correct += (
            preds == y_batch
        ).sum().item()

        total += y_batch.size(0)

    return (
        total_loss / len(loader),
        correct / total
    )


# ============================================================
# LOAD BEST MODEL
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_scores = []

best_models = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X, y)
):

    print(
        f"\n========== FOLD {fold+1} =========="
    )

    X_train = X[train_idx]
    y_train = y[train_idx]

    X_val = X[val_idx]
    y_val = y[val_idx]

    train_dataset = LeviathanDataset(
        X_train,
        y_train
    )

    val_dataset = LeviathanDataset(
        X_val,
        y_val
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=256,
        shuffle=False
    )

    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )

    weights = torch.tensor(
        weights,
        dtype=torch.float32
    ).to(device)

    model = BiLSTMAttention().to(device)

    criterion = nn.CrossEntropyLoss(
        weight=weights,
        label_smoothing=0.05
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=50
    )

    best_acc = 0

    patience = 10

    counter = 0

    best_state = None

    for epoch in range(50):

        train_loss, train_acc = train_epoch(
            model,
            train_loader,
            optimizer,
            criterion
        )

        val_acc = validate(
            model,
            val_loader
        )

        scheduler.step()

        print(
            f"Fold {fold+1} "
            f"Epoch {epoch+1:02d} "
            f"Train {train_acc:.4f} "
            f"Val {val_acc:.4f}"
        )

        if val_acc > best_acc:

            best_acc = val_acc

            best_state = copy.deepcopy(
                model.state_dict()
            )

            counter = 0

        else:

            counter += 1

        if counter >= patience:

            print(
                f"Early Stopping Fold {fold+1}"
            )

            break

    model.load_state_dict(
        best_state
    )

    best_models.append(model)

    fold_scores.append(
        best_acc
    )

    print(
        f"Best Fold Accuracy: {best_acc:.4f}"
    )
    print("\nFold Scores:")

for score in fold_scores:

    print(score)

print(
    "\nMean CV Accuracy:",
    np.mean(fold_scores)
)

print(
    "Std:",
    np.std(fold_scores)
)

(300000, 28)
   sequence_id  timestep  sensor_0  sensor_1  sensor_2  sensor_3  sensor_4  \
0            0         1  0.797988  4.043916  0.855621  4.557594  8.341861   
1            0         2  3.542572  1.454165  4.397192  3.334686  3.895221   
2            0         3  1.825843  4.878263  4.508513  0.489621  2.400877   
3            0         4  2.315402  2.839605  4.443792  1.602136  8.462071   
4            0         5  6.792719  2.199233  3.239968  1.119951  8.425677   

   sensor_5  sensor_6  sensor_7  ...  sensor_16  sensor_17  sensor_18  \
0  2.861494  0.153825  0.245543  ...   0.021982   1.568427   0.211843   
1  0.946766  4.982987  4.270716  ...   0.572948   2.600176   0.869560   
2  1.201121  4.050717  1.707504  ...   1.174360   1.718682   1.042756   
3  2.723718  3.727529  2.289420  ...   2.201044   2.512720   1.623894   
4  0.652766  3.595458  0.895690  ...   0.256178   1.849288   1.598129   

   sensor_19  sensor_20  sensor_21  sensor_22  sensor_23  sensor_24  level  
0 

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import copy

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# ============================================================
# 3. LOAD DATA & BUILD SEQUENCES
# ============================================================

print("Loading data...")
train = pd.read_csv("train.csv")

sensor_cols = [c for c in train.columns if c.startswith("sensor_")]

X, y = [], []
for seq_id, group in train.groupby("sequence_id"):
    group = group.sort_values("timestep")
    X.append(group[sensor_cols].values)
    y.append(int(group["level"].iloc[0]))

X = np.array(X)
y = np.array(y)

print(f"X shape: {X.shape}") # Expected: (N, 10, 25)
print(f"y shape: {y.shape}") # Expected: (N,)

# ============================================================
# 4. SCALE FEATURES
# ============================================================

scaler = StandardScaler()
X_flat = X.reshape(-1, 25)
X_flat = scaler.fit_transform(X_flat)
X = X_flat.reshape(X.shape)

# ============================================================
# 5. DATASET WITH AUGMENTATION
# ============================================================

class LeviathanDataset(Dataset):
    def __init__(self, X, y=None, is_train=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None
        self.is_train = is_train

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_val = self.X[idx]

        # Strategic Augmentation: Inject slight noise during training to prevent overfitting
        if self.is_train:
            noise = torch.randn_like(x_val) * 0.02
            x_val = x_val + noise

        if self.y is None:
            return x_val

        return x_val, self.y[idx]

# ============================================================
# 6. INVERTED TRANSFORMER MODEL
# ============================================================

class InvertedTransformer(nn.Module):
    def __init__(self, seq_len=10, num_sensors=25, d_model=128, nhead=8, num_layers=3, num_classes=4):
        super().__init__()

        # 1. Project the raw 10-timestep sequence of EACH sensor into a hidden dimension
        self.feature_projector = nn.Linear(seq_len, d_model)

        # 2. Sensor Embeddings: Gives the model "spatial" awareness of which sensor is which
        self.sensor_embeddings = nn.Parameter(torch.randn(1, num_sensors, d_model))

        # 3. CLS Token: A learnable token to aggregate all sensor information
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # 4. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            batch_first=True,
            dropout=0.3,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 5. Classification Head
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # Initial shape: (Batch, 10, 25)

        # INVERT: Swap the time and sensor dimensions
        x = x.transpose(1, 2) # New shape: (Batch, 25, 10)

        # Project each sensor's 10-step history into d_model
        x = self.feature_projector(x) # New shape: (Batch, 25, d_model)

        # Add spatial/sensor embeddings
        x = x + self.sensor_embeddings

        # Prepend the CLS token
        B = x.size(0)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) # Shape: (Batch, 26, d_model)

        # Pass through the Transformer
        x = self.transformer(x)

        # Extract the processed CLS token (at index 0)
        cls_out = x[:, 0, :]

        # Classify
        return self.mlp_head(cls_out)

# ============================================================
# 7. FOCAL LOSS
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

# ============================================================
# 8. DEVICE SETUP
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# 9. TRAINING & VALIDATION FUNCTIONS
# ============================================================

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / len(loader), correct / total

def validate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            preds.extend(outputs.argmax(1).cpu().numpy())
            labels.extend(y_batch.numpy())

    return accuracy_score(labels, preds)

# ============================================================
# 10. K-FOLD CROSS VALIDATION LOOP
# ============================================================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
best_models = []
EPOCHS = 50

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n========== FOLD {fold+1} ==========")

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]

    # Notice `is_train=True` enables the Gaussian noise augmentation
    train_dataset = LeviathanDataset(X_train, y_train, is_train=True)
    val_dataset = LeviathanDataset(X_val, y_val, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

    # Class weights for Focal Loss
    weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
    weights = torch.tensor(weights, dtype=torch.float32).to(device)

    # Initialize Model, Loss, Optimizer
    model = InvertedTransformer().to(device)
    criterion = FocalLoss(alpha=weights, gamma=2.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

    # Linear Warmup for the first 5 epochs, followed by Cosine Annealing
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=5)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS-5)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

    best_acc = 0
    patience = 10
    counter = 0
    best_state = None

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
        val_acc = validate(model, val_loader)

        # Step the unified scheduler
        scheduler.step()

        print(f"Fold {fold+1} Epoch {epoch+1:02d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Loss: {train_loss:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print(f"Early Stopping Triggered on Fold {fold+1}!")
            break

    model.load_state_dict(best_state)
    best_models.append(model)
    fold_scores.append(best_acc)
    print(f"Best Fold Accuracy: {best_acc:.4f}")

print("\n============================================")
print("FINAL RESULTS")
print("============================================")
for i, score in enumerate(fold_scores):
    print(f"Fold {i+1}: {score:.4f}")
print(f"\nMean CV Accuracy: {np.mean(fold_scores):.4f}")
print(f"Standard Deviation: {np.std(fold_scores):.4f}")

Loading data...
X shape: (30000, 10, 25)
y shape: (30000,)
Using device: cuda

========== FOLD 1 ==========
Fold 1 Epoch 01 | Train Acc: 0.2227 | Val Acc: 0.2013 | Loss: 0.7954
Fold 1 Epoch 02 | Train Acc: 0.2174 | Val Acc: 0.2012 | Loss: 0.7903
Fold 1 Epoch 03 | Train Acc: 0.2197 | Val Acc: 0.2112 | Loss: 0.7886
Fold 1 Epoch 04 | Train Acc: 0.2170 | Val Acc: 0.2215 | Loss: 0.7899
Fold 1 Epoch 05 | Train Acc: 0.2372 | Val Acc: 0.2502 | Loss: 0.7855
Fold 1 Epoch 06 | Train Acc: 0.2977 | Val Acc: 0.4598 | Loss: 0.7496
Fold 1 Epoch 07 | Train Acc: 0.4794 | Val Acc: 0.6185 | Loss: 0.5734
Fold 1 Epoch 08 | Train Acc: 0.5788 | Val Acc: 0.7392 | Loss: 0.4376
Fold 1 Epoch 09 | Train Acc: 0.6364 | Val Acc: 0.7492 | Loss: 0.3548
Fold 1 Epoch 10 | Train Acc: 0.6784 | Val Acc: 0.7607 | Loss: 0.3067
Fold 1 Epoch 11 | Train Acc: 0.6963 | Val Acc: 0.7877 | Loss: 0.2799
Fold 1 Epoch 12 | Train Acc: 0.7184 | Val Acc: 0.8083 | Loss: 0.2519
Fold 1 Epoch 13 | Train Acc: 0.7233 | Val Acc: 0.7403 | Loss: 0.

In [ ]:
from sklearn.metrics import confusion_matrix

model.eval()

preds = []
labels = []

with torch.no_grad():

    for X_batch, y_batch in val_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        pred = outputs.argmax(1)

        preds.extend(pred.cpu().numpy())
        labels.extend(y_batch.numpy())

cm = confusion_matrix(labels, preds)

print(cm)

[[1228  280   24   57]
 [  11 1134   63    0]
 [   0  139 1211   17]
 [ 150    1  284 1401]]


In [ ]:
# ============================================================
# LOAD TEST DATA
# ============================================================

test = pd.read_csv("test.csv")

print(test.shape)
print(test.head())

# ============================================================
# BUILD TEST SEQUENCES
# ============================================================

X_test = []
test_ids = []

for seq_id, group in test.groupby("sequence_id"):

    group = group.sort_values("timestep")

    X_test.append(
        group[sensor_cols].values
    )

    test_ids.append(seq_id)

X_test = np.array(X_test)

print("X_test shape:", X_test.shape)

# ============================================================
# SCALE USING TRAIN SCALER
# ============================================================

X_test_flat = X_test.reshape(-1, 25)

X_test_flat = scaler.transform(
    X_test_flat
)

X_test = X_test_flat.reshape(
    X_test.shape
)

print("Scaled X_test:", X_test.shape)

(100000, 27)
   sequence_id  timestep  sensor_0  sensor_1  sensor_2  sensor_3   sensor_4  \
0        30000         1  6.359172  6.184699  6.100020  1.732549   7.881978   
1        30000         2  1.890584  3.558338  0.067958  4.487240   6.813329   
2        30000         3  4.330377  2.763814  5.301354  3.381588   5.089594   
3        30000         4  2.308128  0.720017  4.731803  1.056460  10.700831   
4        30000         5  3.290882  3.537784  3.359703  2.514456   1.917834   

   sensor_5  sensor_6  sensor_7  ...  sensor_15  sensor_16  sensor_17  \
0  3.746634  0.698426  1.901331  ...   2.935482   1.658815   1.422796   
1  3.386298  4.718844  2.301154  ...   3.314421   0.579993   0.047388   
2  1.790298  6.612131  0.067361  ...   0.203675   0.714142   2.656040   
3  1.382467  5.661991  1.518023  ...   2.818238   0.862381   0.841426   
4  3.356466  5.880298  1.242493  ...   3.758585   0.884645   2.247270   

   sensor_18  sensor_19  sensor_20  sensor_21  sensor_22  sensor_23  sens

In [ ]:
# ============================================================
# TEST DATASET
# ============================================================

test_dataset = LeviathanDataset(
    X_test,
    y=None,
    is_train=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False
)

In [ ]:
# ============================================================
# ENSEMBLE PREDICTION
# ============================================================

all_probs = []

for fold, model in enumerate(best_models):

    print(f"Predicting Fold {fold+1}")

    model.eval()

    probs = []

    with torch.no_grad():

        for X_batch in test_loader:

            X_batch = X_batch.to(device)

            outputs = model(X_batch)

            prob = torch.softmax(
                outputs,
                dim=1
            )

            probs.append(
                prob.cpu().numpy()
            )

    probs = np.vstack(probs)

    all_probs.append(probs)

# ============================================================
# AVERAGE PROBABILITIES
# ============================================================

final_probs = np.mean(
    all_probs,
    axis=0
)

predictions = np.argmax(
    final_probs,
    axis=1
)

print("Prediction shape:", predictions.shape)

Predicting Fold 1
Predicting Fold 2
Predicting Fold 3
Predicting Fold 4
Predicting Fold 5
Prediction shape: (10000,)


In [ ]:
# ============================================================
# CREATE SUBMISSION
# ============================================================

submission = pd.DataFrame({

    "sequence_id": test_ids,

    "level": predictions

})

print(submission.head())

submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv saved successfully!")

   sequence_id  level
0        30000      2
1        30001      3
2        30002      3
3        30003      3
4        30004      3
submission.csv saved successfully!


In [ ]:
submission = pd.read_csv("submission.csv")

print(submission.head())
print(submission.shape)
print(submission["level"].value_counts())

   sequence_id  level
0        30000      2
1        30001      3
2        30002      3
3        30003      3
4        30004      3
(10000, 2)
level
3    2883
0    2444
2    2340
1    2333
Name: count, dtype: int64


In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import copy

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# ============================================================
# 3. LOAD DATA
# ============================================================

print("Loading data...")

train = pd.read_csv("train.csv")

sensor_cols = [
    c for c in train.columns
    if c.startswith("sensor_")
]

# ============================================================
# 4. BUILD SEQUENCES
# ============================================================

X = []
y = []

for seq_id, group in train.groupby("sequence_id"):

    group = group.sort_values("timestep")

    X.append(
        group[sensor_cols].values
    )

    y.append(
        int(group["level"].iloc[0])
    )

X = np.array(X)
y = np.array(y)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# ============================================================
# 5. SCALE FEATURES
# ============================================================

scaler = StandardScaler()

X_flat = X.reshape(-1, 25)

X_flat = scaler.fit_transform(X_flat)

X = X_flat.reshape(X.shape)

# ============================================================
# 6. DATASET
# ============================================================

class LeviathanDataset(Dataset):

    def __init__(
        self,
        X,
        y=None,
        is_train=False
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        ) if y is not None else None

        self.is_train = is_train

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        x_val = self.X[idx]

        # small gaussian noise augmentation
        if self.is_train:

            noise = torch.randn_like(x_val) * 0.01

            x_val = x_val + noise

        if self.y is None:

            return x_val

        return x_val, self.y[idx]

# ============================================================
# 7. MIXUP
# ============================================================

def mixup_data(x, y, alpha=0.4):

    lam = np.random.beta(alpha, alpha)

    batch_size = x.size(0)

    index = torch.randperm(batch_size).to(device)

    mixed_x = lam * x + (1 - lam) * x[index]

    y_a = y
    y_b = y[index]

    return mixed_x, y_a, y_b, lam

def mixup_criterion(
    criterion,
    pred,
    y_a,
    y_b,
    lam
):

    return (
        lam * criterion(pred, y_a)
        +
        (1 - lam) * criterion(pred, y_b)
    )

# ============================================================
# 8. ENHANCED INVERTED TRANSFORMER
# ============================================================

class EnhancedInvertedTransformer(nn.Module):

    def __init__(
        self,
        seq_len=10,
        num_sensors=25,
        d_model=256,
        nhead=8,
        num_layers=4,
        num_classes=4
    ):

        super().__init__()

        # feature projection
        self.feature_projector = nn.Sequential(

            nn.Linear(seq_len, d_model),

            nn.GELU(),

            nn.Dropout(0.1),

            nn.Linear(d_model, d_model)
        )

        # sensor embeddings
        self.sensor_embeddings = nn.Parameter(

            torch.randn(
                1,
                num_sensors,
                d_model
            ) * 0.02
        )

        # cls token
        self.cls_token = nn.Parameter(

            torch.randn(
                1,
                1,
                d_model
            ) * 0.02
        )

        # transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(

            d_model=d_model,

            nhead=nhead,

            dim_feedforward=d_model * 4,

            dropout=0.2,

            activation='gelu',

            batch_first=True,

            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(

            encoder_layer,

            num_layers=num_layers
        )

        # attention pooling
        self.attention_pool = nn.Sequential(

            nn.Linear(d_model, 1)
        )

        # classification head
        self.head = nn.Sequential(

            nn.LayerNorm(d_model),

            nn.Linear(d_model, 128),

            nn.GELU(),

            nn.Dropout(0.2),

            nn.Linear(128, 64),

            nn.GELU(),

            nn.Dropout(0.1),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):

        # (B, 10, 25) -> (B, 25, 10)
        x = x.transpose(1, 2)

        # feature projection
        x = self.feature_projector(x)

        # add sensor embeddings
        x = x + self.sensor_embeddings

        # cls token
        B = x.size(0)

        cls_tokens = self.cls_token.expand(
            B,
            -1,
            -1
        )

        x = torch.cat(
            (cls_tokens, x),
            dim=1
        )

        # transformer
        x = self.transformer(x)

        # attention pooling
        attn_weights = torch.softmax(

            self.attention_pool(x),

            dim=1
        )

        pooled = (
            x * attn_weights
        ).sum(dim=1)

        # classify
        return self.head(pooled)

# ============================================================
# 9. DEVICE
# ============================================================

device = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Using device: {device}")

# ============================================================
# 10. TRAINING FUNCTIONS
# ============================================================

def train_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        # MIXUP
        X_batch, y_a, y_b, lam = mixup_data(
            X_batch,
            y_batch
        )

        outputs = model(X_batch)

        loss = mixup_criterion(
            criterion,
            outputs,
            y_a,
            y_b,
            lam
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

        preds = outputs.argmax(1)

        correct += (preds == y_batch).sum().item()

        total += y_batch.size(0)

    return (
        total_loss / len(loader),
        correct / total
    )

def validate(
    model,
    loader
):

    model.eval()

    preds = []
    labels = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)

            outputs = model(X_batch)

            preds.extend(
                outputs.argmax(1).cpu().numpy()
            )

            labels.extend(
                y_batch.numpy()
            )

    return accuracy_score(
        labels,
        preds
    )

# ============================================================
# 11. CROSS VALIDATION
# ============================================================

skf = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42
)

fold_scores = []

best_models = []

EPOCHS = 80

# ============================================================
# 12. TRAINING LOOP
# ============================================================

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X, y)
):

    print(f"\n========== FOLD {fold+1} ==========")

    X_train = X[train_idx]
    y_train = y[train_idx]

    X_val = X[val_idx]
    y_val = y[val_idx]

    train_dataset = LeviathanDataset(
        X_train,
        y_train,
        is_train=True
    )

    val_dataset = LeviathanDataset(
        X_val,
        y_val,
        is_train=False
    )

    train_loader = DataLoader(

        train_dataset,

        batch_size=256,

        shuffle=True
    )

    val_loader = DataLoader(

        val_dataset,

        batch_size=256,

        shuffle=False
    )

    # MODEL
    model = EnhancedInvertedTransformer().to(device)

    # LOSS
    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.05
    )

    # OPTIMIZER
    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=3e-4,

        weight_decay=5e-5
    )

    # SCHEDULERS
    warmup = torch.optim.lr_scheduler.LinearLR(

        optimizer,

        start_factor=0.1,

        total_iters=5
    )

    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(

        optimizer,

        T_max=EPOCHS - 5
    )

    scheduler = torch.optim.lr_scheduler.SequentialLR(

        optimizer,

        schedulers=[warmup, cosine],

        milestones=[5]
    )

    # EARLY STOPPING
    best_acc = 0

    patience = 15

    counter = 0

    best_state = None

    # TRAIN
    for epoch in range(EPOCHS):

        train_loss, train_acc = train_epoch(

            model,
            train_loader,
            optimizer,
            criterion
        )

        val_acc = validate(
            model,
            val_loader
        )

        scheduler.step()

        print(

            f"Fold {fold+1} "
            f"Epoch {epoch+1:02d} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Loss: {train_loss:.4f}"
        )

        if val_acc > best_acc:

            best_acc = val_acc

            best_state = copy.deepcopy(
                model.state_dict()
            )

            counter = 0

        else:

            counter += 1

        if counter >= patience:

            print(
                f"Early Stopping "
                f"Triggered on Fold {fold+1}"
            )

            break

    model.load_state_dict(best_state)

    best_models.append(model)

    fold_scores.append(best_acc)

    print(
        f"Best Fold Accuracy: "
        f"{best_acc:.4f}"
    )

# ============================================================
# 13. FINAL RESULTS
# ============================================================

print("\n===================================")
print("FINAL RESULTS")
print("===================================")

for i, score in enumerate(fold_scores):

    print(f"Fold {i+1}: {score:.4f}")

print(
    f"\nMean CV Accuracy: "
    f"{np.mean(fold_scores):.4f}"
)

print(
    f"Standard Deviation: "
    f"{np.std(fold_scores):.4f}"
)

Loading data...
X shape: (30000, 10, 25)
y shape: (30000,)
Using device: cuda

========== FOLD 1 ==========


/tmp/ipykernel_610/2966908022.py:229: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Fold 1 Epoch 01 | Train Acc: 0.3058 | Val Acc: 0.3058 | Loss: 1.3764
Fold 1 Epoch 02 | Train Acc: 0.3030 | Val Acc: 0.3058 | Loss: 1.3763
Fold 1 Epoch 03 | Train Acc: 0.3047 | Val Acc: 0.3058 | Loss: 1.3761
Fold 1 Epoch 04 | Train Acc: 0.3050 | Val Acc: 0.3058 | Loss: 1.3759
Fold 1 Epoch 05 | Train Acc: 0.3052 | Val Acc: 0.3058 | Loss: 1.3761
Fold 1 Epoch 06 | Train Acc: 0.3056 | Val Acc: 0.3058 | Loss: 1.3760
Fold 1 Epoch 07 | Train Acc: 0.3059 | Val Acc: 0.3058 | Loss: 1.3759
Fold 1 Epoch 08 | Train Acc: 0.3059 | Val Acc: 0.3058 | Loss: 1.3758
Fold 1 Epoch 09 | Train Acc: 0.3061 | Val Acc: 0.3058 | Loss: 1.3754
Fold 1 Epoch 10 | Train Acc: 0.3054 | Val Acc: 0.3058 | Loss: 1.3757
Fold 1 Epoch 11 | Train Acc: 0.3060 | Val Acc: 0.3058 | Loss: 1.3756
Fold 1 Epoch 12 | Train Acc: 0.3060 | Val Acc: 0.3058 | Loss: 1.3756
Fold 1 Epoch 13 | Train Acc: 0.3060 | Val Acc: 0.3058 | Loss: 1.3751
Fold 1 Epoch 14 | Train Acc: 0.3039 | Val Acc: 0.3058 | Loss: 1.3751
Fold 1 Epoch 15 | Train Acc: 0.305

/tmp/ipykernel_610/2966908022.py:229: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Fold 2 Epoch 01 | Train Acc: 0.3040 | Val Acc: 0.3060 | Loss: 1.3761
Fold 2 Epoch 02 | Train Acc: 0.3053 | Val Acc: 0.3060 | Loss: 1.3762
Fold 2 Epoch 03 | Train Acc: 0.3060 | Val Acc: 0.3060 | Loss: 1.3759
Fold 2 Epoch 04 | Train Acc: 0.3050 | Val Acc: 0.3060 | Loss: 1.3759
Fold 2 Epoch 05 | Train Acc: 0.3043 | Val Acc: 0.3060 | Loss: 1.3760
Fold 2 Epoch 06 | Train Acc: 0.3061 | Val Acc: 0.3060 | Loss: 1.3758
Fold 2 Epoch 07 | Train Acc: 0.3060 | Val Acc: 0.3060 | Loss: 1.3754
Fold 2 Epoch 08 | Train Acc: 0.3056 | Val Acc: 0.3060 | Loss: 1.3753
Fold 2 Epoch 09 | Train Acc: 0.3055 | Val Acc: 0.3060 | Loss: 1.3751
Fold 2 Epoch 10 | Train Acc: 0.3060 | Val Acc: 0.3060 | Loss: 1.3756
Fold 2 Epoch 11 | Train Acc: 0.3057 | Val Acc: 0.3060 | Loss: 1.3753
Fold 2 Epoch 12 | Train Acc: 0.3061 | Val Acc: 0.2930 | Loss: 1.3746


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import copy

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# ============================================================
# 3. LOAD DATA & BUILD SEQUENCES
# ============================================================

print("Loading data...")
train = pd.read_csv("train.csv")

sensor_cols = [c for c in train.columns if c.startswith("sensor_")]

X, y = [], []
for seq_id, group in train.groupby("sequence_id"):
    group = group.sort_values("timestep")
    X.append(group[sensor_cols].values)
    y.append(int(group["level"].iloc[0]))

X = np.array(X)
y = np.array(y)

print(f"X shape: {X.shape}") # Expected: (N, 10, 25)
print(f"y shape: {y.shape}") # Expected: (N,)

# ============================================================
# 4. SCALE FEATURES
# ============================================================

scaler = StandardScaler()
X_flat = X.reshape(-1, 25)
X_flat = scaler.fit_transform(X_flat)
X = X_flat.reshape(X.shape)

# ============================================================
# 5. DATASET WITH AUGMENTATION
# ============================================================

class LeviathanDataset(Dataset):
    def __init__(self, X, y=None, is_train=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None
        self.is_train = is_train

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_val = self.X[idx]

        # Strategic Augmentation: Inject slight noise during training to prevent overfitting
        if self.is_train:
            noise = torch.randn_like(x_val) * 0.01
            x_val = x_val + noise

        if self.y is None:
            return x_val

        return x_val, self.y[idx]

# ============================================================
# 6. INVERTED TRANSFORMER MODEL
# ============================================================

class InvertedTransformer(nn.Module):
    def __init__(self, seq_len=10, num_sensors=25, d_model=128, nhead=8, num_layers=3, num_classes=4):
        super().__init__()

        # 1. Project the raw 10-timestep sequence of EACH sensor into a hidden dimension
        self.feature_projector = nn.Linear(seq_len, d_model)

        # 2. Sensor Embeddings: Gives the model "spatial" awareness of which sensor is which
        self.sensor_embeddings = nn.Parameter(torch.randn(1, num_sensors, d_model))

        # 3. CLS Token: A learnable token to aggregate all sensor information
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))

        # 4. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            batch_first=True,
            dropout=0.3,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 5. Classification Head
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # Initial shape: (Batch, 10, 25)

        # INVERT: Swap the time and sensor dimensions
        x = x.transpose(1, 2) # New shape: (Batch, 25, 10)

        # Project each sensor's 10-step history into d_model
        x = self.feature_projector(x) # New shape: (Batch, 25, d_model)

        # Add spatial/sensor embeddings
        x = x + self.sensor_embeddings

        # Prepend the CLS token
        B = x.size(0)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) # Shape: (Batch, 26, d_model)

        # Pass through the Transformer
        x = self.transformer(x)

        # Extract the processed CLS token (at index 0)
        cls_out = x[:, 0, :]

        # Classify
        return self.mlp_head(cls_out)

# ============================================================
# 7. FOCAL LOSS
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

# ============================================================
# 8. DEVICE SETUP
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# 9. TRAINING & VALIDATION FUNCTIONS
# ============================================================

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / len(loader), correct / total

def validate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            preds.extend(outputs.argmax(1).cpu().numpy())
            labels.extend(y_batch.numpy())

    return accuracy_score(labels, preds)

# ============================================================
# 10. K-FOLD CROSS VALIDATION LOOP
# ============================================================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []
best_models = []
EPOCHS = 50

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n========== FOLD {fold+1} ==========")

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]

    # Notice `is_train=True` enables the Gaussian noise augmentation
    train_dataset = LeviathanDataset(X_train, y_train, is_train=True)
    val_dataset = LeviathanDataset(X_val, y_val, is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

    # Class weights for Focal Loss
    weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
    weights = torch.tensor(weights, dtype=torch.float32).to(device)

    # Initialize Model, Loss, Optimizer
    model = InvertedTransformer().to(device)
    criterion = FocalLoss(alpha=weights, gamma=2.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)

    # Linear Warmup for the first 5 epochs, followed by Cosine Annealing
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=5)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS-5)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

    best_acc = 0
    patience = 10
    counter = 0
    best_state = None

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
        val_acc = validate(model, val_loader)

        # Step the unified scheduler
        scheduler.step()

        print(f"Fold {fold+1} Epoch {epoch+1:02d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Loss: {train_loss:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print(f"Early Stopping Triggered on Fold {fold+1}!")
            break

    model.load_state_dict(best_state)
    best_models.append(model)
    fold_scores.append(best_acc)
    print(f"Best Fold Accuracy: {best_acc:.4f}")

print("\n============================================")
print("FINAL RESULTS")
print("============================================")
for i, score in enumerate(fold_scores):
    print(f"Fold {i+1}: {score:.4f}")
print(f"\nMean CV Accuracy: {np.mean(fold_scores):.4f}")
print(f"Standard Deviation: {np.std(fold_scores):.4f}")

Loading data...
X shape: (30000, 10, 25)
y shape: (30000,)
Using device: cuda

========== FOLD 1 ==========
Fold 1 Epoch 01 | Train Acc: 0.2228 | Val Acc: 0.2013 | Loss: 0.7954
Fold 1 Epoch 02 | Train Acc: 0.2174 | Val Acc: 0.2012 | Loss: 0.7903
Fold 1 Epoch 03 | Train Acc: 0.2195 | Val Acc: 0.2115 | Loss: 0.7886
Fold 1 Epoch 04 | Train Acc: 0.2171 | Val Acc: 0.2213 | Loss: 0.7899
Fold 1 Epoch 05 | Train Acc: 0.2371 | Val Acc: 0.2502 | Loss: 0.7855
Fold 1 Epoch 06 | Train Acc: 0.2975 | Val Acc: 0.4477 | Loss: 0.7492
Fold 1 Epoch 07 | Train Acc: 0.4809 | Val Acc: 0.5962 | Loss: 0.5718
Fold 1 Epoch 08 | Train Acc: 0.5743 | Val Acc: 0.7193 | Loss: 0.4489
Fold 1 Epoch 09 | Train Acc: 0.6274 | Val Acc: 0.6972 | Loss: 0.3693
Fold 1 Epoch 10 | Train Acc: 0.6748 | Val Acc: 0.7933 | Loss: 0.3124
Fold 1 Epoch 11 | Train Acc: 0.6965 | Val Acc: 0.7950 | Loss: 0.2794
Fold 1 Epoch 12 | Train Acc: 0.7067 | Val Acc: 0.7287 | Loss: 0.2612
Fold 1 Epoch 13 | Train Acc: 0.7258 | Val Acc: 0.7908 | Loss: 0.

KeyboardInterrupt: 